# 🟣 Calenda — Gemma 4 E2B QAT 비교 트랙 (Colab · 단일 GPU)

Qwen3-0.6B(c3)와 **동일 데이터·동일 golden 54**로 비교하기 위한 대안 베이스.

- 베이스: **google/gemma-4-E2B-it-qat-q4_0-unquantized** (QAT 비양자화형 → LoRA SFT → q4_0 GGUF 재양자화 시 QAT 품질 보존)
- 온디바이스: E2B(유효 ~2B) int4 ≈ **~1GB** (폰 예산 ~1GB 가정)
- 표준 파이프라인 재사용: `train_lora.py`·`eval_model.py`가 model_config에서 `response_template`(`<start_of_turn>model\n`)·`supports_system:false`를 읽어 Gemma 대응.

**실행 전:**
1. `런타임 → 런타임 유형 변경 → L4 GPU` (E2B는 ~2B라 L4/A100 권장)
2. ⚠️ **Gemma는 게이트 모델** — HF에서 라이선스 동의 + 좌측 🔑 보안 비밀에 `HF_TOKEN` 등록 필수.

> 선택: 더 빠른 학습은 Unsloth(`unsloth/gemma-4-E2B-it-qat`)로도 가능. 정확한 model id·API는 Unsloth Gemma 4 문서 확인. 이 노트북은 검증된 표준 경로로 동작.

## 1. GPU 확인 (`L4` 권장)

In [ ]:
!nvidia-smi

## 2. repo clone + HF 로그인 (Gemma 게이트 — HF_TOKEN 필수)

In [ ]:
import os
%cd /content
if not os.path.exists('Calenda'):
    !git clone https://github.com/sooryong/Calenda.git
else:
    !cd Calenda && git fetch origin && git reset --hard origin/main
%cd /content/Calenda
# HF_TOKEN (Gemma 게이트 다운로드에 필요)
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = os.environ['HUGGING_FACE_HUB_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN: 설정됨')
except Exception as e:
    print('⚠️ HF_TOKEN 없음 — Gemma 게이트 다운로드 실패함. 좌측 🔑에 HF_TOKEN 등록 후 재실행:', e)

## 3. 설정 + 의존성 (Gemma 4 인식 위해 transformers 업그레이드)

In [ ]:
CONFIG = 'configs/train_gemma4_e2b.yaml'
!pip install -q -e .[train]
!pip install -q -U transformers   # Gemma 4 아키텍처 인식
!pip uninstall torchao -y -q
import os, yaml, torch
os.environ['WANDB_DISABLED'] = 'true'
_cfg  = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))
BASE = _mcfg['hf_id']; NAME = os.path.basename(_cfg['output_dir'])
GOLDEN = _cfg.get('eval_golden', 'data/eval/golden.jsonl')
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
import transformers; print('transformers', transformers.__version__)
print('★ 라운드', _cfg['run_name'], '| base', BASE)
print('  supports_system', _mcfg.get('supports_system'), '| response_template', repr(_mcfg.get('response_template')))
print('  train', _cfg['train_data'], '| golden', GOLDEN)

## 4. 사전점검 — Gemma 챗 템플릿 정렬 + response_template 존재
`OK`면 학습 진행. (system은 Gemma 미지원이라 첫 user 턴에 합쳐짐)

In [ ]:
from transformers import AutoTokenizer
import json, yaml, re
_tok = AutoTokenizer.from_pretrained(BASE)
_sys = _mcfg['system_prompt']
_user = '<채널: KakaoTalk>' + chr(10) + '<메시지>' + chr(10) + '내일 3시 회의' + chr(10) + '</메시지>'
_gold = json.dumps({'schedule_status': 'no', 'events': []}, ensure_ascii=False)
# supports_system=false → system을 첫 user 턴에 접두 (train_lora.py와 동일 렌더)
_uc = _sys + chr(10)+chr(10) + _user
_full = [{'role':'user','content':_uc}, {'role':'assistant','content':_gold}]
_pre  = [{'role':'user','content':_uc}]
_train     = _tok.apply_chat_template(_full, tokenize=False, add_generation_prompt=False)
_user_only = _tok.apply_chat_template(_pre,  tokenize=False, add_generation_prompt=False)
_infer     = _tok.apply_chat_template(_pre,  tokenize=False, add_generation_prompt=True)
# 실제 model 턴 헤더 = generation prompt가 추가한 부분 (베이스 무관 자동 검출)
_marker = _infer[len(_user_only):]
print('자동 검출 response_template:', repr(_marker))
# config 값과 다르면 config 파일을 실제 마커로 교정(read→write) 후 reload → train_lora가 올바른 마커 사용
_cfgp = _cfg['model_config']
if _mcfg.get('response_template') != _marker:
    _s = open(_cfgp, encoding='utf-8').read()
    _newline = 'response_template: ' + json.dumps(_marker, ensure_ascii=False)
    if re.search(r'^response_template:.*$', _s, flags=re.M):
        _s = re.sub(r'^response_template:.*$', _newline, _s, count=1, flags=re.M)
    else:
        _s = _s.rstrip() + chr(10) + _newline + chr(10)
    open(_cfgp, 'w', encoding='utf-8').write(_s)
    _mcfg = yaml.safe_load(open(_cfgp, encoding='utf-8'))
    print('→ config.response_template 자동 교정:', repr(_mcfg['response_template']))
_rt = _mcfg['response_template']
_aligned = _train.startswith(_infer)
_json_next = (_train[len(_infer):] if _aligned else '').lstrip().startswith('{')
print('정렬:', _aligned, '| JSON 직후:', _json_next, '| 마커 존재:', _rt in _train)
assert _aligned and _json_next and _rt in _train, '템플릿 점검 실패 — _infer 끝부분 확인: ' + repr(_infer[-40:])
print('✅ 통과 — 학습 셀 진행 (마커', repr(_rt), ')')

## 5. 학습 (단일 GPU · torchrun 아님). 첫 logging에서 loss 하강 확인.

In [ ]:
print(f'학습 시작 → {CONFIG} ({_cfg["run_name"]}) · 단일 GPU')
!python scripts/train_lora.py --config {CONFIG}

## 6. 평가 (merge + 골든셋). `--model_config`로 Gemma system_prompt·supports_system 사용.

In [ ]:
import os
MERGED_DIR = f'models/merged/{NAME}'; EVAL_JSON = f'logs/eval_{NAME}.json'
!python scripts/merge_lora.py --base {BASE} --lora {_cfg['output_dir']} --out {MERGED_DIR}
!python scripts/eval_model.py --model {MERGED_DIR} --eval {GOLDEN} --out {EVAL_JSON} --model_config {_cfg['model_config']}
print(f'--- {EVAL_JSON} ---')
print(open(EVAL_JSON, encoding='utf-8').read())

## 7. 다운로드 (어댑터 + eval)
양자화는 **calendar_quantize.ipynb**에서 별도. 단 Gemma QAT는 **Q4_0**로 익스포트해야 QAT 품질 보존(Q8_0/Q4_K_M 아님). quantize 노트북의 양자화 레벨을 `['Q4_0']`로 바꿔 실행.
배포 비교: Qwen3-0.6B Q8_0(~600MB) vs Gemma4 E2B Q4_0(~1GB) — 같은 golden 54의 final_score/time_match로 판단.

In [ ]:
import shutil, os, glob
ZIP = f'/content/lora_{NAME}.zip'
shutil.make_archive(ZIP[:-4], 'zip', _cfg['output_dir'])
print('zip', ZIP, os.path.getsize(ZIP)//1024//1024, 'MB')
from google.colab import files
files.download(ZIP); files.download(EVAL_JSON)